In [3]:
### CÉLULA 1: SETUP COMPLETO (LIMPEZA + CARREGAMENTO) ###
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("--- 🚀 Iniciando Etapa 4: Otimização (Modo Autônomo) ---")

# 1. Tenta encontrar o dataset ORIGINAL (já que o limpo sumiu)
arquivos_originais = [
    'ecommerce-sales.csv', 
    'ecommerce_sales.csv',
    '../ecommerce_sales.csv',
    'notebooks/ecommerce_sales.csv'
]

df = None
for arquivo in arquivos_originais:
    if os.path.exists(arquivo):
        print(f"✅ Arquivo ORIGINAL encontrado: {arquivo}")
        df = pd.read_csv(arquivo)
        break

if df is None:
    print("\n⚠️ ERRO CRÍTICO: Nem o arquivo original foi encontrado.")
    print("👉 AÇÃO NECESSÁRIA: Arraste o arquivo 'ecommerce-sales.csv' para a pasta deste notebook no VS Code!")
else:
    print("⚙️ Recriando limpeza da Etapa 2 automaticamente...")
    
    # --- REPLICAÇÃO DA LIMPEZA DA ETAPA 2 ---
    # 1. Definir Alvo
    target_col = 'monthly_sales'
    
    # 2. Remover Duplicatas
    df = df.drop_duplicates()
    
    # 3. Tratamento básico de Nulos (Mediana para numéricos, Moda para texto)
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(exclude=np.number).columns:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    # 4. Remover colunas de ID (Prevenção de Data Leakage)
    cols_to_drop = [col for col in df.columns if 'sale_id' in col or 'id' in col.lower()]
    if target_col in cols_to_drop: cols_to_drop.remove(target_col)
    
    # 5. Encoding (Texto -> Número)
    df_clean = pd.get_dummies(df.drop(columns=cols_to_drop), drop_first=True)
    
    # 6. Preparar X e y
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]
    
    # 7. Divisão 60/20/20
    X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.40, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.50, random_state=42)
    
    print(f"\n✅ Dados processados e prontos!")
    print(f"Treino: {X_train.shape} | Validação: {X_val.shape}")

--- 🚀 Iniciando Etapa 4: Otimização (Modo Autônomo) ---
✅ Arquivo ORIGINAL encontrado: ecommerce_sales.csv
⚙️ Recriando limpeza da Etapa 2 automaticamente...

✅ Dados processados e prontos!
Treino: (1506, 58) | Validação: (502, 58)


In [6]:
### CÉLULA 2: OTIMIZAÇÃO DE HIPERPARÂMETROS (GRID SEARCH) ###

print("--- 🤖 Iniciando Grid Search (Buscando o melhor modelo)... ---")

# 1. Definir o Modelo: Ridge Regression
# (É uma regressão linear melhorada que evita overfitting)
modelo_base = Ridge()

# 2. Definir a "Grade" de parâmetros para testar
# O computador vai testar TODAS essas combinações
param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0], # Força da regularização
    'solver': ['auto', 'svd', 'cholesky', 'lsqr'] # Diferentes métodos matemáticos
}

# 3. Configurar o Grid Search
# cv=5: Divide o treino em 5 partes e valida cruzado (muito robusto)
# scoring='r2': Queremos maximizar o R²
grid_search = GridSearchCV(
    estimator=modelo_base,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1 # Usa todos os núcleos do processador para ir rápido
)

# 4. Treinar (Isso vai rodar o modelo dezenas de vezes!)
grid_search.fit(X_train, y_train)

# 5. Resultados
melhor_modelo = grid_search.best_estimator_

print("\n✅ Otimização Concluída!")
print(f"Melhores Parâmetros: {grid_search.best_params_}")
print(f"Melhor R² (na validação interna): {grid_search.best_score_:.4f}")

--- 🤖 Iniciando Grid Search (Buscando o melhor modelo)... ---
Fitting 5 folds for each of 24 candidates, totalling 120 fits

✅ Otimização Concluída!
Melhores Parâmetros: {'alpha': 100.0, 'solver': 'svd'}
Melhor R² (na validação interna): 0.2719


In [8]:
### CÉLULA EXTRA: RECUPERAR O BASELINE ###
from sklearn.linear_model import LinearRegression

print("🔄 Recalculando o Baseline (Regressão Linear) para comparação justa...")

# 1. Treinar o modelo simples novamente
modelo_simples = LinearRegression()
modelo_simples.fit(X_train, y_train)

# 2. Gerar as previsões do Baseline
pred_baseline = modelo_simples.predict(X_val)
r2_baseline = r2_score(y_val, pred_baseline)
mae_baseline = mean_absolute_error(y_val, pred_baseline)

print(f"✅ Baseline Recuperado!")
print(f"   R² Baseline: {r2_baseline:.4f}")
print(f"   MAE Baseline: {mae_baseline:.4f}")

# Agora temos a variável 'r2_baseline' com o valor correto para usar na comparação!

🔄 Recalculando o Baseline (Regressão Linear) para comparação justa...
✅ Baseline Recuperado!
   R² Baseline: 0.2388
   MAE Baseline: 505.8474


In [13]:
### CÉLULA 3: AVALIAÇÃO FINAL (COM CÁLCULO FORÇADO DO BASELINE) ###

import joblib
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

print("--- 🏆 Comparação Final: Baseline vs Otimizado ---")

# 1. Avaliar o Modelo Otimizado (Ridge)
# (Usa o 'melhor_modelo' que já foi treinado na Célula 2)
y_pred_otimizado = melhor_modelo.predict(X_val)
r2_otimizado = r2_score(y_val, y_pred_otimizado)

# 2. CALCULAR O BASELINE AGORA (Para garantir que não seja zero!)
print("🔄 Treinando Baseline (Regressão Linear) agora mesmo para comparação...")
modelo_linear = LinearRegression()
modelo_linear.fit(X_train, y_train)
y_pred_linear = modelo_linear.predict(X_val)
r2_baseline = r2_score(y_val, y_pred_linear)

# 3. Mostrar a Batalha dos Modelos
print(f"\n📊 PLACAR FINAL (R² na Validação):")
print(f"==================================================")
print(f"📉 Modelo Baseline (Linear): {r2_baseline:.4f}")
print(f"📈 Modelo Otimizado (Ridge): {r2_otimizado:.4f}")
print(f"==================================================")

melhoria = r2_otimizado - r2_baseline

if melhoria > 0.0001:
    print(f"🚀 SUCESSO! O modelo otimizado venceu por {melhoria*100:.2f} pontos percentuais.")
elif melhoria > -0.0001:
    print(f"⚖️ Empate técnico! O Grid Search escolheu parâmetros muito próximos do linear.")
else:
    print(f"🤔 O Baseline venceu. Isso acontece quando a regularização atrapalha modelos simples.")

# 4. Salvar o Modelo Campeão (Ouro 🥇)
if not os.path.exists('../models'):
    os.makedirs('../models', exist_ok=True)

caminho_modelo = '../models/modelo_final.joblib'
joblib.dump(melhor_modelo, caminho_modelo)
print(f"\n💾 Modelo Final salvo com sucesso em: {caminho_modelo}")
print("--- FIM DA ETAPA 4 ---")

--- 🏆 Comparação Final: Baseline vs Otimizado ---
🔄 Treinando Baseline (Regressão Linear) agora mesmo para comparação...

📊 PLACAR FINAL (R² na Validação):
📉 Modelo Baseline (Linear): 0.2388
📈 Modelo Otimizado (Ridge): 0.2545
🚀 SUCESSO! O modelo otimizado venceu por 1.56 pontos percentuais.

💾 Modelo Final salvo com sucesso em: ../models/modelo_final.joblib
--- FIM DA ETAPA 4 ---
